# Analysis of Model Performance

The performance of both models is evaluated using standard metrics such as accuracy, precision, recall, and F1-score.

Accuracy measures the overall correctness of the model, while precision and recall provide insights into class-wise performance. 
F1-score balances precision and recall, making it suitable for evaluating performance on imbalanced datasets.

In [ ]:
## Testing and Evaluation Pipeline , import required files and libraries
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
project_path = Path("/content/drive/MyDrive/ISL_GAN_FewShot_Project")

landmark_dir = project_path / "data" / "landmarks"
augmented_dir = project_path / "data" / "augmented"

baseline_model_candidates = [
    project_path / "models" / "baseline" / "baseline_mlp_best.pth",
    project_path / "models" / "baseline_mlp_best.pth"
]

final_model_candidates = [
    project_path / "models" / "fewshot" / "final_augmented_mlp_best.pth",
    project_path / "models" / "final_augmented_mlp_best.pth"
]

report_dir = project_path / "results"
report_dir.mkdir(parents=True, exist_ok=True)

with open(landmark_dir / "class_mapping.json", "r") as f:
    mapping = json.load(f)

class_to_idx = mapping["class_to_idx"]
idx_to_class = {int(k): v for k, v in mapping["idx_to_class"].items()}
num_classes = len(class_to_idx)

print("Project ready.")
print("Number of classes:", num_classes)

In [ ]:
X_test = np.load(landmark_dir / "X_test.npy")
y_test = np.load(landmark_dir / "y_test.npy")

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
class LandmarkDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

test_dataset = LandmarkDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(BaselineMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(BaselineMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def find_existing_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these model paths exist: {candidates}")

def evaluate_model(model, loader, device):
    model.eval()
    all_true = []
    all_pred = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)

            all_true.extend(y_batch.numpy())
            all_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(all_true, all_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true, all_pred, average="macro", zero_division=0
    )
    return acc, precision, recall, f1, all_true, all_pred

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_test.shape[1]

baseline_model_path = find_existing_path(baseline_model_candidates)
final_model_path = find_existing_path(final_model_candidates)

baseline_model = BaselineMLP(input_dim=input_dim, num_classes=num_classes).to(device)
final_model = BaselineMLP(input_dim=input_dim, num_classes=num_classes).to(device)

baseline_model.load_state_dict(torch.load(baseline_model_path, map_location=device))
final_model.load_state_dict(torch.load(final_model_path, map_location=device))

print("Baseline model loaded from:", baseline_model_path)
print("Final model loaded from:", final_model_path)

In [ ]:
baseline_acc, baseline_prec, baseline_rec, baseline_f1, y_true_base, y_pred_base = evaluate_model(
    baseline_model, test_loader, device
)

final_acc, final_prec, final_rec, final_f1, y_true_final, y_pred_final = evaluate_model(
    final_model, test_loader, device
)

print("Baseline Accuracy:", baseline_acc)
print("Final Accuracy:", final_acc)

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Baseline Model", "GAN-Augmented Model"],
    "Accuracy": [baseline_acc, final_acc],
    "Precision": [baseline_prec, final_prec],
    "Recall": [baseline_rec, final_rec],
    "F1-Score": [baseline_f1, final_f1]
})

comparison_df

In [ ]:
plt.figure(figsize=(10, 6))
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]

x = np.arange(len(metrics))
width = 0.35

baseline_vals = [baseline_acc, baseline_prec, baseline_rec, baseline_f1]
final_vals = [final_acc, final_prec, final_rec, final_f1]

plt.bar(x - width/2, baseline_vals, width, label="Baseline")
plt.bar(x + width/2, final_vals, width, label="GAN-Augmented")

plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("Baseline vs GAN-Augmented Model Comparison")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("Final Model Classification Report")
print(classification_report(
    y_true_final,
    y_pred_final,
    target_names=[idx_to_class[i] for i in range(num_classes)],
    zero_division=0
))

In [ ]:
cm_base = confusion_matrix(y_true_base, y_pred_base)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_base,
    cmap="Blues",
    xticklabels=[idx_to_class[i] for i in range(num_classes)],
    yticklabels=[idx_to_class[i] for i in range(num_classes)],
    annot=False
)
plt.title("Baseline Model Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
cm_final = confusion_matrix(y_true_final, y_pred_final)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_final,
    cmap="Greens",
    xticklabels=[idx_to_class[i] for i in range(num_classes)],
    yticklabels=[idx_to_class[i] for i in range(num_classes)],
    annot=False
)
plt.title("GAN-Augmented Model Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
report = {
    "baseline": {
        "accuracy": float(baseline_acc),
        "precision": float(baseline_prec),
        "recall": float(baseline_rec),
        "f1_score": float(baseline_f1)
    },
    "gan_augmented": {
        "accuracy": float(final_acc),
        "precision": float(final_prec),
        "recall": float(final_rec),
        "f1_score": float(final_f1)
    }
}

report_file = report_dir / "final_comparison_report.json"
with open(report_file, "w") as f:
    json.dump(report, f, indent=4)

comparison_csv = report_dir / "final_comparison_table.csv"
comparison_df.to_csv(comparison_csv, index=False)

print("Saved report to:", report_file)
print("Saved comparison table to:", comparison_csv)

In [ ]:
pred_summary = pd.DataFrame({
    "true_label": y_true_final,
    "predicted_label": y_pred_final,
    "true_class": [idx_to_class[i] for i in y_true_final],
    "predicted_class": [idx_to_class[i] for i in y_pred_final]
})

pred_file = report_dir / "final_predictions_summary.csv"
pred_summary.to_csv(pred_file, index=False)

print("Saved prediction summary to:", pred_file)
pred_summary.head()